# Step 1: Install Dependencies

In [ ]:
!pip install -q transformers accelerate bitsandbytes matplotlib seaborn
!curl -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o cloudflared
!chmod +x cloudflared

# Step 2: Load Nayari

In [ ]:
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_id = "Crossie/Nayari"

print("Loading model weights...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

# Initialize pipeline for easier testing
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

# Step 3: Benchmark Tests

In [ ]:
import time

def run_benchmark():
    tests = {
        "Logic": "If I have 3 apples and you take away 2, how many apples do you have?",
        "Coding": "Write a Python function to check if a number is prime.",
        "Creative": "Write a short haiku about a futuristic robot named Nayari.",
        "Reasoning": "Sally has 3 brothers. Each brother has 2 sisters. How many sisters does Sally have?"
    }
    
    results = {}
    
    print("🚀 Starting Benchmarks...")
    
    for category, prompt in tests.items():
        start_time = time.time()
        
        # Tokenize input
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        input_len = inputs['input_ids'].shape[1]
        
        # Generate
        output_tokens = model.generate(
            **inputs, 
            max_new_tokens=150, 
            temperature=0.7,
            do_sample=True
        )
        
        end_time = time.time()
        
        # Calculations
        duration = end_time - start_time
        new_tokens = output_tokens[0][input_len:]
        num_tokens = len(new_tokens)
        tps = num_tokens / duration
        
        response_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
        
        results[category] = {
            "tps": round(tps, 2),
            "response": response_text,
            "tokens": num_tokens
        }
        print(f"✅ {category} Complete ({round(tps, 2)} tokens/sec)")

    return results

# Run it
benchmark_data = run_benchmark()

# Step 4: Export Graph as JPG

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def save_benchmark_graph(results):
    categories = list(results.keys())
    tps_values = [results[c]['tps'] for c in categories]
    
    plt.figure(figsize=(12, 7))
    sns.set_style("whitegrid")
    
    # Create Bar Plot
    colors = sns.color_palette("magma", len(categories))
    bars = plt.bar(categories, tps_values, color=colors)
    
    plt.title("Crossie/Nayari Model Performance Benchmark", fontsize=16)
    plt.ylabel("Tokens Per Second (TPS)", fontsize=12)
    plt.xlabel("Test Category", fontsize=12)
    
    # Add values on top of bars
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + 0.5, yval, ha='center', va='bottom', fontweight='bold')
    
    # Save as JPG
    plt.savefig("benchmark_results.jpg", format='jpg', dpi=300, bbox_inches='tight')
    print("📊 Graph saved as 'benchmark_results.jpg'")
    plt.show()

save_benchmark_graph(benchmark_data)

# Step 5: Output Text Logs

In [ ]:
with open("model_outputs.txt", "w") as f:
    for cat, data in benchmark_data.items():
        f.write(f"=== {cat} (TPS: {data['tps']}) ===\n")
        f.write(f"Response: {data['response']}\n\n")
print("📝 Outputs saved to 'model_outputs.txt'")

# Step 6: Cloudflare Tunnel (External Download)

In [ ]:
import subprocess
import os
import time

# Start HTTP Server
server_port = 8888
subprocess.Popen(["python", "-m", "http.server", str(server_port)])

# Start Cloudflare
print("Starting Cloudflare Tunnel...")
with open("cf.log", "w") as f:
    p = subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://localhost:{server_port}"], stdout=f, stderr=f)

time.sleep(5)

# Extract URL from log
with open("cf.log", "r") as f:
    log = f.read()
    import re
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log)
    if match:
        print(f"\n✨ ACCESS YOUR FILES HERE:")
        print(f"URL: {match.group(0)}")
        print(f"Click the link and look for 'benchmark_results.jpg'")
    else:
        print("Tunnel URL not found yet, wait a moment and check 'cf.log'")